<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Working with APIs

*Session 3 · Notebook 04 · Lecture · Student version*

## Overview

An **API** lets your code pull data from another service over the web: market data, reference data, public statistics and much more. This notebook shows how to call web APIs from Python with the `requests` library, read the JSON they return, turn it into a pandas DataFrame, handle errors, and page through large results. We use a movie API (which needs a key) and the World Bank API (no key).

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what an API is and read an API's documentation.
- Send a GET request with `requests` and check the status code.
- Parse a JSON response and navigate nested data.
- Turn a JSON response into a pandas DataFrame.
- Recognise common error codes and page through multi-page results.

## Prerequisites

- Session 1 Python (functions, dictionaries, loops).
- Session 2 pandas (building and viewing DataFrames).
- An API key in `config.py` (see Section 2).

## Index

- [Section 1: What is an API?](#what)
- [Section 2: Setup and API keys](#setup)
- [Section 3: Query basics](#query)
- [Section 4: Inspecting the response](#inspect)
- [Section 5: Navigating the JSON](#navigate)
- [Section 6: From JSON to a DataFrame](#frame)
- [Section 7: Bad queries and error codes](#bad)
- [Section 8: Pagination](#pagination)
- [Key Takeaways](#takeaways)
- [Further Reading](#reading)

<a id="what"></a>
# Section 1: What is an API?

An **API** (Application Programming Interface) is a way for one program to request data or services from another over the web. You send an HTTP **request** to a URL (an *endpoint*), and the server sends back a **response**, usually as **JSON** (nested dictionaries and lists).

**Common HTTP methods:**

- **GET** read data (almost all data APIs).
- **POST** send or create data.
- **PUT / PATCH** update; **DELETE** remove.

**Status-code families** (the response tells you what happened):

- **2xx** success (200 OK).
- **4xx** your request was wrong (400 bad request, 401 unauthorised, 404 not found).
- **5xx** the server failed.

> **The golden rule:** read the API's documentation first. Every API names its parameters and structures its response differently.

<a id="setup"></a>
# Section 2: Setup and API keys

Install `requests` if you need it (`python -m pip install requests`), then import the stack.

**API keys** can cost money per call and must stay private, so we keep ours in a `config.py` file in this folder and import it, rather than typing the key into the notebook.

In [ ]:
# Dependencies
import requests
from pprint import pprint
import json
import pandas as pd

In [ ]:
# Import the API key from config.py (in this folder)
from config import movies_api_key

print(movies_api_key)

<a id="query"></a>
# Section 3: Query basics

We use the [OMDb movie API](http://www.omdbapi.com/) (see the [requests docs](https://docs.python-requests.org/en/latest/)). A query is just a URL with parameters: here `?t=` is the movie title and `&apikey=` is your key.

In [ ]:
##Validate your API key:
api_key = movies_api_key

In [ ]:
##Movie Search
##Setting the parameters for the search
url = 'http://www.omdbapi.com/?t=' # ?t= refers to the title of the movie
movie = 'Casino Royale' ## Change to any movie of your preference
#movie = 'UP'
api_key_str = "&apikey=" + api_key

print('call to send to the API:', url + movie + api_key_str)

In [ ]:
response = requests.get(url + movie + api_key_str)
response
##If you get "<Response [200]>" then everything is fine, otherwise review your code, specially your API key.

In [ ]:
##Print the queried url. NOTE: Make sure to comment out as it will also print the API key
print(response.url)

### Checking the response

You can paste the printed URL into a browser to see the same result. Back in Python, the **status code** tells you if it worked: `200` means success.

In [ ]:
# Check status code. A 200 means everything is ok with the syntax of the query.
response.status_code

**2xx codes** mean the server received and processed the request successfully:

- **200 OK**: the request succeeded.
- **201 Created**: a resource was created (often after a POST).
- **204 No Content**: success, but nothing is returned.

In [ ]:
response.encoding

<a id="inspect"></a>
# Section 4: Inspecting the response

The `response` object carries the raw content, the server headers, and a handy `.json()` method that converts the body into a Python dictionary.

In [ ]:
##Accessing the content of the response:
response.content

In [ ]:
# Server's information for this request
response.headers

In [ ]:
# Convert the response to JSON format
data = response.json()
print(type(data))

In [ ]:
##Pprint helps us to visualise better a JSON/dictionary data
pprint(data)
#print(data)

In [ ]:
# Retriveve all the Keys
data.keys()

In [ ]:
# Retriveve all Values
data.values()

<a id="navigate"></a>
# Section 5: Navigating the JSON

Because the response is now a dictionary, we read values by key. Some values are themselves lists of dictionaries (here, `Ratings`), which we index into.

In [ ]:
# Get individual Responses from the JSON file
director = data['Director']
box_office = data['BoxOffice']

In [ ]:
print(f'Movie director {director}')
print(f'Amount made at the box office {box_office}')

In [ ]:
# From multiple values
## Lets analyse the "rating" of the movie
####First level:
print(type(data['Ratings']))
data['Ratings']

In [ ]:
# From multiple values
print(type(data['Ratings'][0]))
##Accessing the first element of the list:
print(data['Ratings'][0])

In [ ]:
##Accessing the key, value pair:
print(data['Ratings'][0].keys())
##Raiting source
print(data['Ratings'][0]['Source'])
##Raiting:
print(data['Ratings'][0]['Value'])

### Exercise 5.1: Print all the ratings

The `Ratings` value is a list of dictionaries. Print every rating as `Source: Value`.

In [ ]:
# Your turn. Write your solution here:


### Exercise 5.2: A movie-details function

Write a function that asks the user for a movie title, queries the API, and prints the title, director, box office and all ratings (handle a bad title gracefully).

In [ ]:
# Your turn. Write your solution here:


<a id="frame"></a>
# Section 6: From JSON to a DataFrame

API responses are nested JSON, but for analysis we usually want a flat **table**. `pd.json_normalize` flattens a dictionary (or list of dictionaries) into a DataFrame in one step.

In [ ]:
# Flatten the whole movie response into a one-row DataFrame
movie_df = pd.json_normalize(data)
movie_df

In [ ]:
# The nested 'Ratings' list becomes its own tidy table
ratings_df = pd.json_normalize(data['Ratings'])
ratings_df

<a id="bad"></a>
# Section 7: Bad queries and error codes

When a request is malformed or the key is wrong, the API returns an error code and a message instead of data. The OMDb docs say a query must include `t` (title) or `i` (id); omitting it gives a `400`.

In [ ]:
# Look for all movies
url2 = 'http://www.omdbapi.com/?'
bad_key = '1234'
api = 'apikey='
year = "&y="+ '2019'


In [ ]:
## Bad API KEY
#query= url2 + api+ bad_key + year
# Not Formatted properly
query= url2 + api_key + year

In [ ]:
print(query)

In [ ]:
# Performing a GET request 
response2 = requests.get(query)
print(response2)
# Response 400 means it is a bad request

In [ ]:
# Convert the response to JSON format
action_data = response2.json()
pprint(action_data)

**4xx codes** mean the request was wrong:

- **400 Bad Request**: something is wrong in the request.
- **401 Unauthorised**: the client is not allowed to make this request.
- **403 Forbidden**: correct request, but no permission.
- **404 Not Found**: the resource does not exist.

<a id="pagination"></a>
# Section 8: Pagination

**Pagination** limits how many results come back per request, to keep responses manageable. The API's documentation tells you the page size; you then loop over pages to collect everything. We use the [World Bank country API](https://datahelpdesk.worldbank.org/knowledgebase/articles/898590).

In [ ]:
url3 = "http://api.worldbank.org/v2/"
api_format = "json"
query2= f"{url3}countries?format={api_format}"
query2


In [ ]:
###Validating the reply:
requests.get(query2)

In [ ]:
# Get country information in JSON format
countries_response = requests.get(query2).json()
print(type(countries_response), len(countries_response))
countries_response

In [ ]:
# First element is general information, second is countries themselves
print(countries_response[0])
countries = countries_response[1]
countries

In [ ]:
# Check number of responses
print(f'Number of responses {len(countries)}')

The first element of the response is metadata, which tells us how many **pages** there are.

In [ ]:
##We can see that there are more elements to extract:
## 'page': 1, 'pages': 6, 'per_page': '50
countries_response[0]

In [ ]:
##Requesting the second page:
query2= f"{url3}countries?&page={1}&format=json"
query2

In [ ]:
##Testing the amount of pages that are in the data source
##Lets see if there is a page = 20
query2= f"{url3}countries?&page={20}&format=json"
request_pgs = requests.get(query2)
##We can see that the request was successful:
request_pgs
#print(query2, requests.get(query2))
#requests.get(query2).json()[1]

#print(len(requests.get(query2).json()[1]))


In [ ]:
##But accessing the actual data we can see that has no countries info.
print(request_pgs.json()[0])
request_pgs.json()[1]

In [ ]:
requests.get(query2).json()

### Exercise 8.1: Collect every country

Page through the API and collect the information for **all** countries into one list. Do not hardcode the number of pages; stop when a page comes back empty.

In [ ]:
# Your turn. Write your solution here:


### Exercise 8.2: Build a DataFrame

From the collected data, build a DataFrame with each country's `name` and its income-level value.

In [ ]:
# Your turn. Write your solution here:


<a id="takeaways"></a>
## Key Takeaways

| Step | How |
|---|---|
| Send a request | `requests.get(url)` |
| Check it worked | `response.status_code` (200 = OK) |
| Read the body | `data = response.json()` |
| Navigate JSON | index by key; lists by position |
| Make a table | `pd.json_normalize(data)` |
| Handle errors | check the status code; read the message |
| Get everything | loop over pages until empty |

And always: keep API keys in `config.py`, and read the documentation first.

## Conclusion

You can now call a web API, parse its JSON, turn it into a DataFrame, and handle errors and pagination. Practise these skills on three real APIs in the **03_04 API practical**.

<a id="reading"></a>
## Further Reading and Resources

- [requests documentation](https://docs.python-requests.org/en/latest/).
- [OMDb API](http://www.omdbapi.com/) and [World Bank API](https://datahelpdesk.worldbank.org/knowledgebase/articles/898590).
- [pandas: json_normalize](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html).
- [HTTP status codes (MDN)](https://developer.mozilla.org/en-US/docs/Web/HTTP/Status).